<a href="https://colab.research.google.com/github/plasmaade34/liquid-telemetry-research/blob/main/Monte_Carloo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PROJECT AVRO: DYNAMIC TELEMETRY BENCHMARK  v1.1  (CONFIDENTIAL - PRE-FILING)
# Hodgins Holdings - North Bay, Ontario
# Simulation benchmark: fluid-volume estimation under race-car inertial loads
#
# v1.1 changes vs v1.0 (as received 2026-10-17):
#   C1  EKF observation model inverted to match the sensor model (÷cos, not ×cos)
#       and Jacobian re-derived; outlier gate now uses the innovation covariance.
#   C2  Slosh sub-model discretised with symplectic Euler (matches the simulator).
#       v1.0 explicit-Euler matrix had |eig| > 1 at 10 Hz (unstable).
#   C3  Optional 4th state: fluid-height rate (drain rate) removes tracking lag.
#   +   Monte-Carlo robustness study: 20 seeds x 4 IMU-noise levels.
# ==============================================================================
import numpy as np

N, DT = 1000, 0.1
R_V = 37.5
AREA = np.pi * R_V**2
H_TOT = 200.0
XO, YO = 3.0, 1.5
G0 = 9.80665
WN = 2 * np.pi * 1.25
ZETA = 0.035
A1 = 1 - 2 * ZETA * WN * DT


def simulate(seed=42):
    """Ground truth + synthetic VL53L4CD stream. Identical physics to v1.0."""
    rng = np.random.RandomState(seed)
    t = np.linspace(0, (N - 1) * DT, N)
    vol = np.linspace(650.0, 450.0, N)
    h = vol * 1000.0 / AREA
    ax = -1.8 * np.sin(0.05*np.pi*t) * np.clip(np.sin(0.02*np.pi*t), 0, 1)
    ay = 2.1 * np.cos(0.04*np.pi*t) * np.clip(np.cos(0.015*np.pi*t), -1, 1)
    az = -np.ones(N)
    g_eff = np.sqrt(ax**2 + ay**2 + az**2)
    tilt = np.arctan2(np.sqrt(ax**2 + ay**2), np.abs(az))
    f_in = (1.841 / (R_V/1000.0)) * (ay*G0) * 0.001
    th = np.zeros(N); om = np.zeros(N)
    for k in range(1, N):
        om[k] = om[k-1] + (f_in[k-1] - 2*ZETA*WN*om[k-1] - WN**2*th[k-1]) * DT
        th[k] = th[k-1] + om[k] * DT
    d = (H_TOT - h) / np.cos(tilt + th) + XO*np.sin(th) + YO*np.sin(tilt)
    d = d + rng.normal(0, 0.6, N)
    spk = np.zeros(N); idx = rng.choice(N, 25, replace=False)
    spk[idx] = rng.uniform(12, 25, 25)
    d = d + spk
    return dict(t=t, vol=vol, h=h, ax=ax, ay=ay, g_eff=g_eff, tilt=tilt,
                f_in=f_in, d=d, h_cos=H_TOT - d*np.cos(tilt), h_raw=H_TOT - d)


def to_vol(h):
    return np.clip(h * AREA / 1000.0, 0, 800)


def kalman_1d(s):
    z = s["h_cos"]; est = np.zeros(N); est[0] = z[0]; P = 10.0
    for k in range(1, N):
        m = z[k] if abs(z[k] - est[k-1]) <= 12 else est[k-1]
        P += 0.05
        R = 2.0 * (1 + 3*(s["g_eff"][k] - 1)**2)
        K = P / (P + R); est[k] = est[k-1] + K*(m - est[k-1]); P *= (1 - K)
    return est


def ekf_v10(s):
    """v1.0 3-state EKF exactly as received (for the corrections record)."""
    F = np.array([[1, 0, 0], [0, 1, DT], [0, -WN**2*DT, A1]])
    B = np.array([0, 0, DT]); x = np.array([s["h_cos"][0], 0, 0.0])
    P = np.diag([5.0, 0.1, 0.5]); Q = np.diag([0.005, 0.001, 0.05]); out = [x[0]]
    for k in range(1, N):
        xp = F@x + B*s["f_in"][k-1]; Pp = F@P@F.T + Q
        a = s["tilt"][k] + xp[1]; raw = s["d"][k]
        if abs(raw - (H_TOT - xp[0])) > 15: raw = H_TOT - xp[0]
        zh = xp[0]*np.cos(a) - (XO*np.sin(xp[1]) + YO*np.sin(s["tilt"][k]))
        Hm = np.array([[np.cos(a), -xp[0]*np.sin(a) - XO*np.cos(xp[1]), 0]])
        S = (Hm@Pp@Hm.T)[0, 0] + 0.36*(1 + 0.5*(s["g_eff"][k] - 1))
        K = (Pp@Hm.T)/S; x = xp + (K*((H_TOT - raw) - zh)).ravel()
        P = (np.eye(3) - K@Hm)@Pp; out.append(x[0])
    return np.array(out)


def ekf(s, rate=False, tilt=None, f_in=None, Qh=None, Qr=1e-6, gate=4.0):
    """v1.1 EKF. rate=False -> 3-state [h, th, om]; rate=True -> 4-state + dh/dt."""
    tilt = s["tilt"] if tilt is None else tilt
    f_in = s["f_in"] if f_in is None else f_in
    n = 4 if rate else 3
    Qh = (1e-4 if rate else 5e-3) if Qh is None else Qh
    F = np.eye(n)
    F[1, 1] = 1 - WN**2*DT**2; F[1, 2] = DT*A1; F[2, 1] = -WN**2*DT; F[2, 2] = A1
    if rate: F[0, 3] = DT
    B = np.zeros(n); B[1] = DT**2; B[2] = DT
    x = np.zeros(n); x[0] = s["h_cos"][0]
    P = np.diag([5.0, 0.01, 0.05] + ([1.0] if rate else []))
    Q = np.diag([Qh, 1e-6, 1e-3] + ([Qr] if rate else []))
    out = [x[0]]
    for k in range(1, N):
        xp = F@x + B*f_in[k-1]; Pp = F@P@F.T + Q
        h, th = xp[0], xp[1]; a = tilt[k] + th
        d_hat = (H_TOT - h)/np.cos(a) + XO*np.sin(th) + YO*np.sin(tilt[k])
        Hm = np.zeros((1, n))
        Hm[0, 0] = -1/np.cos(a)
        Hm[0, 1] = (H_TOT - h)*np.sin(a)/np.cos(a)**2 + XO*np.cos(th)
        S = (Hm@Pp@Hm.T)[0, 0] + 0.36*(1 + 0.5*(s["g_eff"][k] - 1))
        y = s["d"][k] - d_hat
        if abs(y) > gate*np.sqrt(S):          # splash rejection
            x, P = xp, Pp
        else:
            K = (Pp@Hm.T)/S; x = xp + (K*y).ravel(); P = (np.eye(n) - K@Hm)@Pp
        out.append(x[0])
    return np.array(out)


def metrics(h_est, vol_true):
    e = np.abs(to_vol(h_est) - vol_true)
    return dict(mae=e.mean(), rmse=np.sqrt((e**2).mean()), max=e.max(),
                comp=(e <= 1.0).mean()*100)


def noisy_inputs(s, seed, imu_deg, fin_pct):
    rng = np.random.default_rng(seed + 1000)
    return (s["tilt"] + np.radians(rng.normal(0, imu_deg, N)),
            s["f_in"] * (1 + rng.normal(0, fin_pct, N)))


if __name__ == "__main__":
    s = simulate(42)
    stages = {
        "Raw uncompensated": s["h_raw"],
        "Cosine tilt comp": s["h_cos"],
        "1D Kalman": kalman_1d(s),
        "3-state EKF (v1.0 as received)": ekf_v10(s),
        "3-state EKF (v1.1 corrected)": ekf(s),
        "4-state EKF (v1.1 + drain rate)": ekf(s, rate=True),
    }
    print(f"{'STAGE':34s} {'MAE':>7s} {'RMSE':>7s} {'MAX':>7s} {'<=1mL':>7s}")
    for k, v in stages.items():
        m = metrics(v, s["vol"])
        print(f"{k:34s} {m['mae']:7.2f} {m['rmse']:7.2f} {m['max']:7.2f} {m['comp']:6.1f}%")
    print("\nMonte Carlo (20 seeds): mean <=1 mL compliance [min]")
    for lab, imu, fin in [("ideal", 0, 0), ("0.2 deg", 0.2, .05),
                          ("0.5 deg", 0.5, .10), ("1.0 deg", 1.0, .20)]:
        r = {"1D": [], "3st": [], "4st": []}
        for sd in range(20):
            ss = simulate(sd); tl, fi = noisy_inputs(ss, sd, imu, fin)
            r["1D"].append(metrics(kalman_1d(ss), ss["vol"])["comp"])
            r["3st"].append(metrics(ekf(ss, tilt=tl, f_in=fi), ss["vol"])["comp"])
            r["4st"].append(metrics(ekf(ss, rate=True, tilt=tl, f_in=fi), ss["vol"])["comp"])
        print(f"  IMU {lab:8s} " + "  ".join(f"{k} {np.mean(v):5.1f}% [{np.min(v):5.1f}]" for k, v in r.items()))

STAGE                                  MAE    RMSE     MAX   <=1mL
Raw uncompensated                   178.59  221.38  498.85    0.9%
Cosine tilt comp                      5.90   11.40  104.92   16.0%
1D Kalman                             3.48    4.86   13.01   30.4%
3-state EKF (v1.0 as received)      249.06  256.29  350.00    0.1%
3-state EKF (v1.1 corrected)          1.76    1.85    3.41   11.4%
4-state EKF (v1.1 + drain rate)       0.25    0.36    3.14   98.8%

Monte Carlo (20 seeds): mean <=1 mL compliance [min]
  IMU ideal    1D  33.6% [ 29.3]  3st   6.0% [  3.7]  4st  99.0% [ 91.8]
  IMU 0.2 deg  1D  33.6% [ 29.3]  3st   8.2% [  5.2]  4st  97.8% [ 89.8]
  IMU 0.5 deg  1D  33.6% [ 29.3]  3st  14.4% [  8.5]  4st  88.8% [ 76.7]
  IMU 1.0 deg  1D  33.6% [ 29.3]  3st  10.5% [  5.0]  4st  54.9% [  0.4]
